# ARC2 014 R B4 paired budget diagnosis
Sorokin 4B TTT, frozen base-only kgmon selection, two strategies (R0 train 280s vs R1 train min(480, 700-220)s) on the frozen B4 cohort, two waves with the same task on the same card, 700s per task per strategy.
Normal save: B4 only (4 tasks / 6 queries), <=45 minutes including loading, 180s save reserve. Hidden rerun: all tasks, <=12 hours with 10-minute saving reserve, 013-equivalent adaptive budget; recording-only raw beams never enter selection.
Diagnostic run only. No score improvement is claimed. Raw complete beams are saved separately and do not participate in selection.


In [ ]:
import base64, hashlib, json, os, subprocess, sys, time
from pathlib import Path
notebook_started = time.time()
source_dir = Path('/kaggle/working/014-r-b4-code')
source_dir.mkdir(exist_ok=False)
blobs = json.loads(base64.b64decode('...'))
expected = {
  "arc_loader.py": "8fae269c67f468e704cab380337055e8361f3990e9f542e60e173c2a1b0607aa",
  "arc_solver.py": "b54cb596aaf6cf63f506fe47d9e1a55e86a246628ad3cf3dd40d333c9b7ced4a",
  "build_submission.py": "1d82fea6c2fef557828fe51c59a4cf51e7f9bed38e6c645f61daa9cd38178c4e",
  "supervise.py": "83ac86f79cc54eadecd76ae3c0e2377be1f0b8e3b2fc9b22524791761ae90b86",
  "worker_pool.py": "eff6b573dc9f5f842b10f863d36b3e5e95a1d125ceb0296855b57d5d3dde9c61",
  "b4_cohort.json": "ac233b4ae4e25249fc6a7be884c83d5cccb3c9c422a80030734f2ac80eaf511a"
}
for name, encoded in blobs.items():
    raw = base64.b64decode(encoded)
    assert hashlib.sha256(raw).hexdigest() == expected[name], name
    (source_dir / name).write_bytes(raw)
(source_dir / 'source-manifest.json').write_text(json.dumps(expected, indent=2))
# The frozen upstream Utility Script contains installed packages, not wheels.
# Find its actual mounted root and make its import path explicit for children.
def utility_name(path):
    return 'pip_install_unsloth_flash_patch' in str(path).replace('-', '_')
found = set()
for entry in sys.path:
    candidate = Path(entry or '.')
    if utility_name(candidate) and (candidate / 'unsloth/__init__.py').is_file():
        found.add(str(candidate.resolve()))
for mount in [Path('/kaggle/usr/lib'), Path('/kaggle/input')]:
    if mount.is_dir():
        for path in mount.rglob('unsloth/__init__.py'):
            if utility_name(path):
                found.add(str(path.parent.parent.resolve()))
utility_roots = sorted(found)
assert len(utility_roots) == 1, ('Expected one original Unsloth Utility Script', utility_roots)
utility = utility_roots[0]
env = dict(os.environ)
env['PYTHONPATH'] = os.pathsep.join([utility, str(source_dir), env.get('PYTHONPATH', '')])
env['PYTHONUNBUFFERED'] = '1'
print(json.dumps({'event': 'boot', 'source_sha256': expected, 'utility_root': utility,
                  'python': sys.version, 'notebook_started': notebook_started}), flush=True)
(source_dir / 'boot.json').write_text(json.dumps({'source_sha256': expected,
    'utility_root': utility, 'python': sys.version, 'notebook_started': notebook_started}, indent=2))
result = subprocess.run([sys.executable, '-u', str(source_dir / 'supervise.py'),
                         '--notebook-start', str(notebook_started)], env=env)
print('SUPERVISOR_RETURN_CODE', result.returncode, flush=True)
assert result.returncode == 0, 'Inspect 014-r-b4-run/supervisor.json and wave worker logs'
assert Path('/kaggle/working/submission.json').is_file()
